In [1]:
# ============================================================
# SPAN MARGIN CALCULATOR — STEP 1: Portfolio Parsing & Net Position Mapping
# MCX Commodity Futures | Google Colab Ready
# ============================================================

# ── Imports ──────────────────────────────────────────────────
import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# ============================================================
# 1. RAW PORTFOLIO DATA  (from FUTCOM_FINAL_PORTFOLIO.xlsx)
# ============================================================

portfolio_data = {
    "Commodity"    : ["Turmeric", "Coriander", "Jeera", "Guar Gum",
                      "Guar Seed", "Castor Seed", "Cotton Seed Oil Cake", "Kapas (Cotton)"],
    "Symbol"       : ["TMCFGRNZM", "DHANIYA", "JEERAUNJHA", "GUARGUM5",
                      "GUARSEED10", "CASTOR", "COCUDAKL", "KAPAS"],
    "Sector"       : ["Spices", "Spices", "Spices",
                      "Oilseeds & Derivatives", "Oilseeds & Derivatives",
                      "Oilseeds & Derivatives", "Oilseeds & Derivatives", "Fiber"],
    "Expiry"       : ["20-Apr-2026", "20-Apr-2026", "20-Apr-2026", "20-Apr-2026",
                      "20-Apr-2026", "20-Apr-2026", "20-Apr-2026", "30-Apr-2026"],
    "Close_Rs_Qtl" : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],   # ₹ per quintal
    "Lot_Size_MT"  : [5, 5, 3, 5, 5, 5, 10, 4],                              # metric tonnes per lot
    "Multiplier"   : [50, 50, 30, 50, 50, 50, 100, 40],                       # quintals per lot
    "Contract_Val" : [811000, 667500, 668550, 537800,
                      285750, 319600, 356700, 67280],                          # ₹ per lot
    "Num_Lots"     : [1, 2, 2, 2, 4, 4, 2, 7],
}

df = pd.DataFrame(portfolio_data)

# ============================================================
# 2. STEP 1 COMPUTATIONS
# ============================================================

# --- 2a. Direction & Net Position ---
df["Direction"]    = "LONG"          # all positions are long; change to SHORT / NET as needed
df["Net_Position"] = df["Num_Lots"]  # net lots (+ve = long, -ve = short)

# --- 2b. Lot size in Quintals (1 MT = 10 Qtl) ---
df["Lot_Size_Qtl"] = df["Lot_Size_MT"] * 10

# --- 2c. Verify Contract Value : Price(₹/Qtl) × Lot_size(Qtl) ---
df["CV_Verify"]  = df["Close_Rs_Qtl"] * df["Lot_Size_Qtl"]
df["CV_Match"]   = df["CV_Verify"] == df["Contract_Val"]

# --- 2d. Total Exposure per position ---
df["Total_Exposure"] = df["Contract_Val"] * df["Num_Lots"]

# --- 2e. Net Delta  (for linear futures delta = 1 per lot) ---
df["Net_Delta"] = df["Net_Position"]

# ============================================================
# 3. DISPLAY RESULTS
# ============================================================

print("=" * 70)
print("  SPAN — STEP 1 : NET POSITION MAPPING")
print("=" * 70)

display_cols = ["Commodity", "Symbol", "Sector", "Direction",
                "Num_Lots", "Lot_Size_Qtl", "Close_Rs_Qtl",
                "Contract_Val", "Total_Exposure", "Net_Delta"]

print("\n📋 Position Table:")
print(df[display_cols].to_string(index=False))

# --- Totals ---
total_exposure   = df["Total_Exposure"].sum()
total_invested   = df["Contract_Val"].mul(df["Num_Lots"]).sum()   # same as above

print(f"\n{'─'*50}")
print(f"  Total Portfolio Exposure  : ₹{total_exposure:>15,.0f}")
print(f"  Total Actual Invested     : ₹{total_invested:>15,.0f}")
print(f"{'─'*50}")

# --- Sector-wise Breakdown ---
print("\n📊 Sector-wise Exposure:")
sector_df = (
    df.groupby("Sector")["Total_Exposure"]
    .sum()
    .reset_index()
    .assign(Weight_Pct=lambda x: (x["Total_Exposure"] / x["Total_Exposure"].sum() * 100).round(2))
    .sort_values("Total_Exposure", ascending=False)
)
print(sector_df.to_string(index=False))

# --- Contract Value Verification ---
print("\n✅ Contract Value Verification (Price × Lot_Size_Qtl = Contract_Val):")
verify_cols = ["Commodity", "Close_Rs_Qtl", "Lot_Size_Qtl", "CV_Verify", "Contract_Val", "CV_Match"]
print(df[verify_cols].to_string(index=False))

all_match = df["CV_Match"].all()
print(f"\n  All contract values verified: {'✅ YES' if all_match else '❌ MISMATCH FOUND'}")

# ============================================================
# 4. OUTPUT DATAFRAME  (passed to Step 2)
# ============================================================

# Clean output — keep only columns needed downstream
span_step1 = df[[
    "Commodity", "Symbol", "Sector", "Expiry",
    "Direction", "Net_Position", "Net_Delta",
    "Lot_Size_Qtl", "Close_Rs_Qtl",
    "Contract_Val", "Num_Lots", "Total_Exposure"
]].copy()

print("\n🔁 `span_step1` dataframe ready → passes into Step 2 (SPAN Parameters / PSR)")
print(span_step1.dtypes)

  SPAN — STEP 1 : NET POSITION MAPPING

📋 Position Table:
           Commodity     Symbol                 Sector Direction  Num_Lots  Lot_Size_Qtl  Close_Rs_Qtl  Contract_Val  Total_Exposure  Net_Delta
            Turmeric  TMCFGRNZM                 Spices      LONG         1            50         16220        811000          811000          1
           Coriander    DHANIYA                 Spices      LONG         2            50         13350        667500         1335000          2
               Jeera JEERAUNJHA                 Spices      LONG         2            30         22285        668550         1337100          2
            Guar Gum   GUARGUM5 Oilseeds & Derivatives      LONG         2            50         10756        537800         1075600          2
           Guar Seed GUARSEED10 Oilseeds & Derivatives      LONG         4            50          5715        285750         1143000          4
         Castor Seed     CASTOR Oilseeds & Derivatives      LONG         4    

In [2]:
# ============================================================
# SPAN MARGIN CALCULATOR — STEP 2: SPAN Parameters & Scanning Risk
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span_step1  (DataFrame from Step 1)
# ============================================================

import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)

# ============================================================
# PASTE span_step1 HERE if running standalone
# (skip if running after Step 1 in same session)
# ============================================================

portfolio_data = {
    "Commodity"    : ["Turmeric", "Coriander", "Jeera", "Guar Gum",
                      "Guar Seed", "Castor Seed", "Cotton Seed Oil Cake", "Kapas (Cotton)"],
    "Symbol"       : ["TMCFGRNZM", "DHANIYA", "JEERAUNJHA", "GUARGUM5",
                      "GUARSEED10", "CASTOR", "COCUDAKL", "KAPAS"],
    "Sector"       : ["Spices", "Spices", "Spices",
                      "Oilseeds & Derivatives", "Oilseeds & Derivatives",
                      "Oilseeds & Derivatives", "Oilseeds & Derivatives", "Fiber"],
    "Expiry"       : ["20-Apr-2026", "20-Apr-2026", "20-Apr-2026", "20-Apr-2026",
                      "20-Apr-2026", "20-Apr-2026", "20-Apr-2026", "30-Apr-2026"],
    "Direction"    : ["LONG"] * 8,
    "Net_Position" : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Delta"    : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl" : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl" : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val" : [811000, 667500, 668550, 537800, 285750, 319600, 356700, 67280],
    "Num_Lots"     : [1, 2, 2, 2, 4, 4, 2, 7],
    "Total_Exposure": [811000, 1335000, 1337100, 1075600, 1143000, 1278400, 713400, 470960],
}
span_step1 = pd.DataFrame(portfolio_data)

# ============================================================
# 2A. SPAN PARAMETERS — Price Scan Range (PSR)
# ============================================================
# PSR = the maximum price move MCX assumes over 1 trading day
# Source: MCX SPAN Parameter File (published daily by MCX)
# These are representative values — update from MCX's official
# SPAN file at: https://www.mcxindia.com/market-data/span-file
#
# PSR is expressed as % of closing price (99th percentile move)
# ============================================================

span_params = {
    # Symbol        : PSR%   (Price Scan Range as % of closing price)
    "TMCFGRNZM"    : 5.0,    # Turmeric
    "DHANIYA"      : 5.0,    # Coriander
    "JEERAUNJHA"   : 5.0,    # Jeera
    "GUARGUM5"     : 5.0,    # Guar Gum
    "GUARSEED10"   : 5.0,    # Guar Seed
    "CASTOR"       : 4.0,    # Castor Seed
    "COCUDAKL"     : 4.0,    # Cotton Seed Oil Cake
    "KAPAS"        : 4.0,    # Kapas (Cotton)
}

# ============================================================
# 2B. 16 RISK SCENARIOS (SPAN Standard)
# ============================================================
# SPAN evaluates P&L across 16 scenarios:
#   Scenarios 1–14 : Price moves ±1/3, ±2/3, ±1× PSR
#                    (with Vol up/down — irrelevant for futures)
#   Scenarios 15–16: Extreme moves ±2× PSR, weighted at 35%
#
# For LINEAR FUTURES: P&L = Price_Move × Lot_Size × Num_Lots
# Volatility scan range (VSR) does NOT affect futures P&L
# ============================================================

EXTREME_WEIGHT = 0.35   # MCX standard: 35% weight on extreme scenarios

# Price move fractions for scenarios 1–14
SCENARIO_FRACTIONS = [-1, -2/3, -1/3, 0, 1/3, 2/3, 1]  # × PSR

def build_scenario_table():
    """Returns list of (scenario_id, price_fraction, weight) for all 16."""
    scenarios = []
    sid = 1
    for frac in SCENARIO_FRACTIONS:
        for _ in range(2):   # vol up / vol down (irrelevant for futures)
            scenarios.append((sid, frac, 1.0))
            sid += 1
    # Scenarios 15 & 16 : extreme moves ±2×PSR at 35% weight
    scenarios.append((15,  2.0, EXTREME_WEIGHT))
    scenarios.append((16, -2.0, EXTREME_WEIGHT))
    return scenarios

SCENARIOS = build_scenario_table()

# ============================================================
# 2C. COMPUTE SCANNING RISK PER POSITION
# ============================================================

df = span_step1.copy()

# Attach PSR%
df["PSR_Pct"]     = df["Symbol"].map(span_params)

# PSR in ₹/Qtl  (absolute price move)
df["PSR_Rs_Qtl"]  = df["Close_Rs_Qtl"] * df["PSR_Pct"] / 100

# PSR in ₹/Lot  (= PSR_Rs_Qtl × Lot_Size_Qtl)
df["PSR_Rs_Lot"]  = df["PSR_Rs_Qtl"] * df["Lot_Size_Qtl"]

# ── Scenario P&L matrix ──────────────────────────────────────
# For each position × scenario:
#   P&L = price_move × lot_size_qtl × num_lots
#         where price_move = fraction × PSR_Rs_Qtl
#   For LONG: loss when price moves down  →  P&L = +fraction (negative frac = loss)

pnl_records = []

for _, row in df.iterrows():
    for sid, frac, wt in SCENARIOS:
        price_move = frac * row["PSR_Rs_Qtl"]           # ₹/Qtl move
        # For LONG position: P&L = price_move × lot_size × num_lots
        sign = 1 if row["Direction"] == "LONG" else -1
        pnl  = sign * price_move * row["Lot_Size_Qtl"] * row["Num_Lots"]
        weighted_pnl = pnl * wt                          # 35% weight on extremes
        pnl_records.append({
            "Symbol"      : row["Symbol"],
            "Commodity"   : row["Commodity"],
            "Scenario"    : sid,
            "Frac"        : round(frac, 4),
            "Weight"      : wt,
            "PnL"         : pnl,
            "Weighted_PnL": weighted_pnl,
        })

pnl_df = pd.DataFrame(pnl_records)

# Scanning Risk = worst (most negative) weighted P&L across 16 scenarios
scanning_risk = (
    pnl_df.groupby("Symbol")["Weighted_PnL"]
    .min()                   # most negative = worst loss
    .abs()                   # express as positive margin requirement
    .reset_index()
    .rename(columns={"Weighted_PnL": "Scanning_Risk_Rs"})
)

df = df.merge(scanning_risk, on="Symbol")

# ============================================================
# 3. DISPLAY RESULTS
# ============================================================

print("=" * 70)
print("  SPAN — STEP 2 : SPAN PARAMETERS & SCANNING RISK")
print("=" * 70)

print("\n📐 SPAN Parameters (Price Scan Range):")
param_cols = ["Commodity", "Symbol", "Close_Rs_Qtl", "PSR_Pct",
              "PSR_Rs_Qtl", "PSR_Rs_Lot"]
print(df[param_cols].to_string(index=False))

print("\n\n⚠️  Scanning Risk per Position:")
risk_cols = ["Commodity", "Symbol", "Num_Lots", "Total_Exposure",
             "PSR_Rs_Lot", "Scanning_Risk_Rs"]
print(df[risk_cols].to_string(index=False))

total_scanning = df["Scanning_Risk_Rs"].sum()
total_exposure = df["Total_Exposure"].sum()

print(f"\n{'─'*55}")
print(f"  Total Scanning Risk (before credits)  : ₹{total_scanning:>14,.0f}")
print(f"  Total Portfolio Exposure               : ₹{total_exposure:>14,.0f}")
print(f"  Scanning Risk as % of Exposure         : {total_scanning/total_exposure*100:>10.2f}%")
print(f"{'─'*55}")

# ── Worst Scenario per commodity ──
print("\n\n🔍 Worst Scenario Detail per Commodity:")
worst = (
    pnl_df.sort_values("Weighted_PnL")
    .groupby("Symbol")
    .first()
    .reset_index()[["Symbol", "Scenario", "Frac", "Weight", "Weighted_PnL"]]
    .assign(Loss_Rs=lambda x: x["Weighted_PnL"].abs())
)
print(worst.to_string(index=False))

# ── Full scenario P&L for first commodity (illustration) ──
sample_sym = df["Symbol"].iloc[0]
print(f"\n\n📊 All 16 Scenario P&Ls for {df['Commodity'].iloc[0]} ({sample_sym}):")
print(
    pnl_df[pnl_df["Symbol"] == sample_sym]
    [["Scenario", "Frac", "Weight", "PnL", "Weighted_PnL"]]
    .to_string(index=False)
)

# ============================================================
# 4. OUTPUT DATAFRAME  (passes to Step 3)
# ============================================================

span_step2 = df[[
    "Commodity", "Symbol", "Sector", "Expiry",
    "Direction", "Net_Position", "Net_Delta",
    "Lot_Size_Qtl", "Close_Rs_Qtl",
    "Contract_Val", "Num_Lots", "Total_Exposure",
    "PSR_Pct", "PSR_Rs_Qtl", "PSR_Rs_Lot",
    "Scanning_Risk_Rs"
]].copy()

print("\n\n🔁 `span_step2` dataframe ready → passes into Step 3 (Inter-commodity Spread Credits)")
print(span_step2[["Commodity", "Scanning_Risk_Rs"]].to_string(index=False))


  SPAN — STEP 2 : SPAN PARAMETERS & SCANNING RISK

📐 SPAN Parameters (Price Scan Range):
           Commodity     Symbol  Close_Rs_Qtl  PSR_Pct  PSR_Rs_Qtl  PSR_Rs_Lot
            Turmeric  TMCFGRNZM         16220     5.00      811.00   40,550.00
           Coriander    DHANIYA         13350     5.00      667.50   33,375.00
               Jeera JEERAUNJHA         22285     5.00    1,114.25   33,427.50
            Guar Gum   GUARGUM5         10756     5.00      537.80   26,890.00
           Guar Seed GUARSEED10          5715     5.00      285.75   14,287.50
         Castor Seed     CASTOR          6392     4.00      255.68   12,784.00
Cotton Seed Oil Cake   COCUDAKL          3567     4.00      142.68   14,268.00
      Kapas (Cotton)      KAPAS          1682     4.00       67.28    2,691.20


⚠️  Scanning Risk per Position:
           Commodity     Symbol  Num_Lots  Total_Exposure  PSR_Rs_Lot  Scanning_Risk_Rs
            Turmeric  TMCFGRNZM         1          811000   40,550.00         

In [3]:
# ============================================================
# SPAN MARGIN CALCULATOR — STEP 3: Inter-Commodity Spread Credits
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span_step2  (DataFrame from Step 2)
# ============================================================

import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)

# ============================================================
# PASTE span_step2 HERE if running standalone
# (skip if running after Step 2 in same session)
# ============================================================

portfolio_data = {
    "Commodity"        : ["Turmeric","Coriander","Jeera","Guar Gum",
                          "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"           : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                          "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "Sector"           : ["Spices","Spices","Spices",
                          "Oilseeds & Derivatives","Oilseeds & Derivatives",
                          "Oilseeds & Derivatives","Oilseeds & Derivatives","Fiber"],
    "Expiry"           : ["20-Apr-2026","20-Apr-2026","20-Apr-2026","20-Apr-2026",
                          "20-Apr-2026","20-Apr-2026","20-Apr-2026","30-Apr-2026"],
    "Direction"        : ["LONG"] * 8,
    "Net_Position"     : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Delta"        : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"     : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"     : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"     : [811000, 667500, 668550, 537800, 285750, 319600, 356700, 67280],
    "Num_Lots"         : [1, 2, 2, 2, 4, 4, 2, 7],
    "Total_Exposure"   : [811000, 1335000, 1337100, 1075600, 1143000, 1278400, 713400, 470960],
    "PSR_Pct"          : [5.0, 5.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0],
    "PSR_Rs_Qtl"       : [811.0, 667.5, 1114.25, 537.8, 285.75, 255.68, 142.68, 67.28],
    "PSR_Rs_Lot"       : [40550, 33375, 33427.5, 26890, 14287.5, 12784, 14268, 2691.2],
    "Scanning_Risk_Rs" : [40550, 66750, 66855, 53780, 57150, 51136, 28536, 18838.4],
}
span_step2 = pd.DataFrame(portfolio_data)

# ============================================================
# 3A. INTER-COMMODITY SPREAD PAIRS
# ============================================================
# MCX grants a spread credit when two commodities are economically
# linked (raw material ↔ derivative, or highly correlated crops).
# Credit reduces margin because risks partially offset each other.
#
# Credit Rate: % of the SMALLER leg's scanning risk that is waived
# Source: MCX SPAN Parameter File → "Inter-Commodity Spread Table"
#
# Key pair in this portfolio:
#   Guar Seed (GUARSEED10) ↔ Guar Gum (GUARGUM5)
#   Guar Gum is processed from Guar Seed → very high price correlation
#   MCX Credit Rate: ~75%
#
# Note: Same-direction positions in correlated commodities still get
# credit because combined risk < sum of individual risks (diversification)
# ============================================================

SPREAD_PAIRS = [
    # (Leg_1_Symbol,  Leg_2_Symbol,  Credit_Rate%, Label)
    ("GUARSEED10",  "GUARGUM5",   75.0,  "Guar Seed / Guar Gum"),
    # Add more pairs here as MCX updates its spread table, e.g.:
    # ("CASTOR",      "COCUDAKL",   25.0,  "Castor / Cotton Seed OC"),
]

# ============================================================
# 3B. DELTA MATCHING — How many spreads can be formed?
# ============================================================
# Matched spreads = min(|Net_Delta Leg1|, |Net_Delta Leg2|)
# Credit is applied only to matched portion.
# ============================================================

df = span_step2.copy()

# Index by symbol for easy lookup
sym_index = df.set_index("Symbol")

spread_results = []

for leg1_sym, leg2_sym, credit_rate, label in SPREAD_PAIRS:

    if leg1_sym not in sym_index.index or leg2_sym not in sym_index.index:
        print(f"⚠️  Skipping '{label}' — one or both symbols not in portfolio")
        continue

    leg1 = sym_index.loc[leg1_sym]
    leg2 = sym_index.loc[leg2_sym]

    # Delta of each leg (absolute — same direction still forms an economic spread)
    delta_leg1 = abs(leg1["Net_Delta"])
    delta_leg2 = abs(leg2["Net_Delta"])

    # Number of matched spread pairs
    matched_spreads = min(delta_leg1, delta_leg2)

    # Scanning risk per lot (delta-adjusted)
    sr_per_lot_leg1 = leg1["Scanning_Risk_Rs"] / delta_leg1
    sr_per_lot_leg2 = leg2["Scanning_Risk_Rs"] / delta_leg2

    # Credit per spread = credit_rate × scanning_risk of the SMALLER leg per lot
    smaller_sr_per_lot = min(sr_per_lot_leg1, sr_per_lot_leg2)
    credit_per_spread  = (credit_rate / 100) * smaller_sr_per_lot

    # Total spread credit
    total_credit = matched_spreads * credit_per_spread

    spread_results.append({
        "Spread_Pair"        : label,
        "Leg_1"              : leg1["Commodity"],
        "Leg_2"              : leg2["Commodity"],
        "Credit_Rate_Pct"    : credit_rate,
        "Delta_Leg1"         : delta_leg1,
        "Delta_Leg2"         : delta_leg2,
        "Matched_Spreads"    : matched_spreads,
        "SR_per_lot_Leg1"    : sr_per_lot_leg1,
        "SR_per_lot_Leg2"    : sr_per_lot_leg2,
        "Smaller_SR_per_lot" : smaller_sr_per_lot,
        "Credit_per_Spread"  : credit_per_spread,
        "Total_Credit_Rs"    : total_credit,
    })

spread_df = pd.DataFrame(spread_results)

# Total inter-commodity spread credit
total_ic_credit = spread_df["Total_Credit_Rs"].sum() if not spread_df.empty else 0.0

# ============================================================
# 3C. ADJUSTED SCANNING RISK
# ============================================================
# Adjusted Scanning Risk = Total Scanning Risk − Spread Credit
# (floor at 0 — credit cannot make margin negative)

total_scanning_risk = df["Scanning_Risk_Rs"].sum()
adj_scanning_risk   = max(total_scanning_risk - total_ic_credit, 0.0)

# ============================================================
# 4. DISPLAY RESULTS
# ============================================================

print("=" * 70)
print("  SPAN — STEP 3 : INTER-COMMODITY SPREAD CREDITS")
print("=" * 70)

print("\n📋 Individual Scanning Risks (from Step 2):")
print(df[["Commodity", "Symbol", "Net_Delta", "Scanning_Risk_Rs"]].to_string(index=False))
print(f"\n  Raw Total Scanning Risk : ₹{total_scanning_risk:>14,.2f}")

print("\n\n🔗 Spread Pairs Identified:")
if spread_df.empty:
    print("  No qualifying spread pairs found in this portfolio.")
else:
    disp_cols = ["Spread_Pair", "Leg_1", "Leg_2", "Credit_Rate_Pct",
                 "Matched_Spreads", "Smaller_SR_per_lot",
                 "Credit_per_Spread", "Total_Credit_Rs"]
    print(spread_df[disp_cols].to_string(index=False))

print(f"\n\n{'─'*55}")
print(f"  Raw Total Scanning Risk          : ₹{total_scanning_risk:>12,.2f}")
print(f"  (−) Inter-Commodity Spread Credit: ₹{total_ic_credit:>12,.2f}")
print(f"  {'─'*40}")
print(f"  Adjusted Scanning Risk           : ₹{adj_scanning_risk:>12,.2f}")
print(f"  Margin Reduction from Credits    :  {(total_ic_credit/total_scanning_risk*100):>10.2f}%")
print(f"{'─'*55}")

print("\n\n💡 Notes:")
print("  • Guar Seed → Guar Gum (Guar Gum is processed from Guar Seed)")
print("    High price correlation → MCX awards 75% credit on matched deltas")
print("  • Credit applies to matched delta portion only")
print("    (Leg1 delta=4, Leg2 delta=2 → 2 spreads matched)")
print("  • Unmatched Leg1 deltas (2 lots of Guar Seed) get NO credit")
print("  • Update SPREAD_PAIRS from MCX SPAN file for live rates")

# ============================================================
# 5. OUTPUT DATAFRAME  (passes to Step 4)
# ============================================================

span_step3 = {
    "position_df"          : df,            # full position table
    "spread_df"            : spread_df,     # spread credit detail
    "total_scanning_risk"  : total_scanning_risk,
    "total_ic_credit"      : total_ic_credit,
    "adj_scanning_risk"    : adj_scanning_risk,
}

print(f"\n\n🔁 `span_step3` dict ready → passes into Step 4 (Delivery Month & Calendar Spread Charges)")
print(f"   Keys: {list(span_step3.keys())}")

  SPAN — STEP 3 : INTER-COMMODITY SPREAD CREDITS

📋 Individual Scanning Risks (from Step 2):
           Commodity     Symbol  Net_Delta  Scanning_Risk_Rs
            Turmeric  TMCFGRNZM          1         40,550.00
           Coriander    DHANIYA          2         66,750.00
               Jeera JEERAUNJHA          2         66,855.00
            Guar Gum   GUARGUM5          2         53,780.00
           Guar Seed GUARSEED10          4         57,150.00
         Castor Seed     CASTOR          4         51,136.00
Cotton Seed Oil Cake   COCUDAKL          2         28,536.00
      Kapas (Cotton)      KAPAS          7         18,838.40

  Raw Total Scanning Risk : ₹    383,595.40


🔗 Spread Pairs Identified:
         Spread_Pair     Leg_1    Leg_2  Credit_Rate_Pct  Matched_Spreads  Smaller_SR_per_lot  Credit_per_Spread  Total_Credit_Rs
Guar Seed / Guar Gum Guar Seed Guar Gum            75.00                2           14,287.50          10,715.62        21,431.25


──────────────────────

In [4]:
# ============================================================
# SPAN MARGIN CALCULATOR — STEP 4: Delivery Month Charge &
#                                   Calendar Spread Charge
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span_step3  (dict from Step 3)
# ============================================================

import pandas as pd
import numpy as np
from datetime import date, datetime

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)

# ============================================================
# PASTE span_step3 inputs HERE if running standalone
# ============================================================

portfolio_data = {
    "Commodity"        : ["Turmeric","Coriander","Jeera","Guar Gum",
                          "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"           : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                          "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "Sector"           : ["Spices","Spices","Spices",
                          "Oilseeds & Derivatives","Oilseeds & Derivatives",
                          "Oilseeds & Derivatives","Oilseeds & Derivatives","Fiber"],
    "Expiry"           : ["20-Apr-2026","20-Apr-2026","20-Apr-2026","20-Apr-2026",
                          "20-Apr-2026","20-Apr-2026","20-Apr-2026","30-Apr-2026"],
    "Direction"        : ["LONG"] * 8,
    "Net_Position"     : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Delta"        : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"     : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"     : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"     : [811000, 667500, 668550, 537800, 285750, 319600, 356700, 67280],
    "Num_Lots"         : [1, 2, 2, 2, 4, 4, 2, 7],
    "Total_Exposure"   : [811000, 1335000, 1337100, 1075600, 1143000, 1278400, 713400, 470960],
    "PSR_Pct"          : [5.0, 5.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0],
    "PSR_Rs_Qtl"       : [811.0, 667.5, 1114.25, 537.8, 285.75, 255.68, 142.68, 67.28],
    "PSR_Rs_Lot"       : [40550, 33375, 33427.5, 26890, 14287.5, 12784, 14268, 2691.2],
    "Scanning_Risk_Rs" : [40550, 66750, 66855, 53780, 57150, 51136, 28536, 18838.4],
}
df = pd.DataFrame(portfolio_data)

# From Step 3 outputs
total_scanning_risk = df["Scanning_Risk_Rs"].sum()
total_ic_credit     = 10714.05        # Guar Seed / Guar Gum credit from Step 3
adj_scanning_risk   = total_scanning_risk - total_ic_credit

# ============================================================
# 4A. DATE SETUP
# ============================================================

TODAY = date(2026, 4, 8)    # ← update to run date if needed

def parse_expiry(s):
    return datetime.strptime(s, "%d-%b-%Y").date()

df["Expiry_Date"]    = df["Expiry"].apply(parse_expiry)
df["Days_to_Expiry"] = (df["Expiry_Date"] - TODAY).dt.days \
                        if hasattr((df["Expiry_Date"] - TODAY), "dt") \
                        else df["Expiry_Date"].apply(lambda d: (d - TODAY).days)

# ============================================================
# 4B. DELIVERY MONTH CHARGE (DMC)
# ============================================================
# MCX imposes an ADDITIONAL charge on contracts in or near
# their delivery/expiry month — liquidity thins out and
# price risk (spot convergence risk) increases sharply.
#
# MCX Tiered DMC Schedule (approximate — verify from MCX circulars):
# ┌─────────────────────────────┬──────────────────────┐
# │ Days to Expiry              │ DMC Rate (% of CV)   │
# ├─────────────────────────────┼──────────────────────┤
# │ > 30 days                   │ 0.00%  (no charge)   │
# │ 16 – 30 days                │ 1.00%                │
# │ 6  – 15 days                │ 2.00%                │
# │ 1  –  5 days  (last week)   │ 3.00%                │
# │ 0 days        (expiry day)  │ 5.00%                │
# └─────────────────────────────┴──────────────────────┘
# CV = Contract Value × Num_Lots  (total position value)
# ============================================================

DMC_TIERS = [
    (0,   0,   5.00),   # expiry day
    (1,   5,   3.00),   # last 5 days
    (6,   15,  2.00),   # 6-15 days
    (16,  30,  1.00),   # 16-30 days (expiry month)
    (31,  9999, 0.00),  # beyond expiry month
]

def get_dmc_rate(days):
    for lo, hi, rate in DMC_TIERS:
        if lo <= days <= hi:
            return rate
    return 0.0

df["DMC_Rate_Pct"] = df["Days_to_Expiry"].apply(get_dmc_rate)
df["DMC_Rs"]       = (df["DMC_Rate_Pct"] / 100) * df["Total_Exposure"]

total_dmc = df["DMC_Rs"].sum()

# ============================================================
# 4C. CALENDAR SPREAD CHARGE (CSC)
# ============================================================
# CSC applies when the SAME commodity has positions in
# MULTIPLE expiry months (e.g., Long Apr + Short Jun Guar Seed).
# It charges for the BASIS RISK between the two months.
#
# CSC Rate: % of the near-month contract value per spread formed
# Source: MCX SPAN Parameter File → "Intra-Commodity Spread Table"
#
# ── Portfolio Check ──────────────────────────────────────────
# Each commodity in this portfolio has EXACTLY ONE expiry month.
# ∴ No same-commodity cross-month positions exist.
# ∴ Calendar Spread Charge = ₹0
#
# If you add a second expiry for any symbol, plug it into
# CSC_PAIRS below and this block will auto-compute the charge.
# ============================================================

# CSC rates (₹/spread or % of near-month CV) — MCX published values
CSC_RATES = {
    # "SYMBOL" : rate_pct   # % of near-month contract value per spread
    # e.g. "GUARSEED10" : 0.50,
}

# Build expiry map: symbol → list of (expiry, net_delta, contract_val)
from collections import defaultdict
expiry_map = defaultdict(list)
for _, row in df.iterrows():
    expiry_map[row["Symbol"]].append({
        "expiry"      : row["Expiry_Date"],
        "net_delta"   : row["Net_Delta"],
        "contract_val": row["Contract_Val"],
    })

csc_records = []
for sym, months in expiry_map.items():
    if len(months) < 2:
        continue     # single expiry — no CSC
    months_sorted  = sorted(months, key=lambda x: x["expiry"])
    near, far      = months_sorted[0], months_sorted[1]
    matched        = min(abs(near["net_delta"]), abs(far["net_delta"]))
    rate_pct       = CSC_RATES.get(sym, 0.0)
    csc_per_spread = (rate_pct / 100) * near["contract_val"]
    total_csc      = matched * csc_per_spread
    csc_records.append({
        "Symbol"          : sym,
        "Near_Expiry"     : near["expiry"],
        "Far_Expiry"      : far["expiry"],
        "Matched_Spreads" : matched,
        "CSC_Rate_Pct"    : rate_pct,
        "CSC_Rs"          : total_csc,
    })

csc_df     = pd.DataFrame(csc_records) if csc_records else pd.DataFrame(
                columns=["Symbol","Near_Expiry","Far_Expiry",
                         "Matched_Spreads","CSC_Rate_Pct","CSC_Rs"])
total_csc  = csc_df["CSC_Rs"].sum() if not csc_df.empty else 0.0

# ============================================================
# 4D. FINAL STEP-4 MARGIN
# ============================================================

step4_margin = adj_scanning_risk + total_dmc + total_csc

# ============================================================
# 5. DISPLAY RESULTS
# ============================================================

print("=" * 70)
print("  SPAN — STEP 4 : DELIVERY MONTH CHARGE & CALENDAR SPREAD CHARGE")
print("=" * 70)

print(f"\n📅 Valuation Date : {TODAY}  |  All prices as of market close\n")

print("📋 Delivery Month Charge (DMC) per Position:")
dmc_cols = ["Commodity", "Symbol", "Expiry", "Days_to_Expiry",
            "Total_Exposure", "DMC_Rate_Pct", "DMC_Rs"]
print(df[dmc_cols].to_string(index=False))

print(f"\n  Total DMC : ₹{total_dmc:>12,.2f}")

print("\n\n🔀 Calendar Spread Charge (CSC):")
if csc_df.empty:
    print("  NIL — Each commodity has a single expiry month in this portfolio.")
    print("  No intra-commodity cross-month positions detected.")
else:
    print(csc_df.to_string(index=False))
    print(f"\n  Total CSC : ₹{total_csc:>12,.2f}")

print(f"\n\n{'─'*60}")
print(f"  Adj. Scanning Risk (after IC credits) : ₹{adj_scanning_risk:>12,.2f}")
print(f"  (+) Delivery Month Charge (DMC)        : ₹{total_dmc:>12,.2f}")
print(f"  (+) Calendar Spread Charge (CSC)       : ₹{total_csc:>12,.2f}")
print(f"  {'─'*48}")
print(f"  SPAN Margin after Step 4               : ₹{step4_margin:>12,.2f}")
print(f"{'─'*60}")

print("\n\n⚠️  DMC Tier applied today (Apr 8, 2026):")
for lo, hi, rate in DMC_TIERS:
    tag = " ← YOUR CONTRACTS" if lo <= 12 <= hi else ""
    label = f"{lo}–{hi} days" if hi < 9999 else f"> {lo} days"
    print(f"   {label:20s} → {rate:.2f}%{tag}")

# ============================================================
# 6. OUTPUT DICT  (passes to Step 5)
# ============================================================

span_step4 = {
    "position_df"       : df,
    "csc_df"            : csc_df,
    "total_scanning_risk"  : total_scanning_risk,
    "total_ic_credit"      : total_ic_credit,
    "adj_scanning_risk"    : adj_scanning_risk,
    "total_dmc"            : total_dmc,
    "total_csc"            : total_csc,
    "step4_margin"         : step4_margin,
}

print(f"\n\n🔁 `span_step4` dict ready → passes into Step 5 (Short Option Minimum / Final SPAN Margin)")
print(f"   Keys: {list(span_step4.keys())}")

  SPAN — STEP 4 : DELIVERY MONTH CHARGE & CALENDAR SPREAD CHARGE

📅 Valuation Date : 2026-04-08  |  All prices as of market close

📋 Delivery Month Charge (DMC) per Position:
           Commodity     Symbol      Expiry  Days_to_Expiry  Total_Exposure  DMC_Rate_Pct    DMC_Rs
            Turmeric  TMCFGRNZM 20-Apr-2026              12          811000          2.00 16,220.00
           Coriander    DHANIYA 20-Apr-2026              12         1335000          2.00 26,700.00
               Jeera JEERAUNJHA 20-Apr-2026              12         1337100          2.00 26,742.00
            Guar Gum   GUARGUM5 20-Apr-2026              12         1075600          2.00 21,512.00
           Guar Seed GUARSEED10 20-Apr-2026              12         1143000          2.00 22,860.00
         Castor Seed     CASTOR 20-Apr-2026              12         1278400          2.00 25,568.00
Cotton Seed Oil Cake   COCUDAKL 20-Apr-2026              12          713400          2.00 14,268.00
      Kapas (Cotton)     

In [5]:
# ============================================================
# SPAN MARGIN CALCULATOR — STEP 5: Final SPAN Margin Assembly
#   Short Option Minimum | Net Option Value | Exposure Margin
#   Complete Margin Summary & Portfolio Health Check
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span_step4  (dict from Step 4)
# ============================================================

import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# ============================================================
# PASTE span_step4 inputs HERE if running standalone
# ============================================================

portfolio_data = {
    "Commodity"        : ["Turmeric","Coriander","Jeera","Guar Gum",
                          "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"           : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                          "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "Sector"           : ["Spices","Spices","Spices",
                          "Oilseeds & Derivatives","Oilseeds & Derivatives",
                          "Oilseeds & Derivatives","Oilseeds & Derivatives","Fiber"],
    "Direction"        : ["LONG"] * 8,
    "Num_Lots"         : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Delta"        : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"     : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"     : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"     : [811000, 667500, 668550, 537800, 285750, 319600, 356700, 67280],
    "Total_Exposure"   : [811000, 1335000, 1337100, 1075600, 1143000, 1278400, 713400, 470960],
    "PSR_Pct"          : [5.0, 5.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0],
    "Scanning_Risk_Rs" : [40550, 66750, 66855, 53780, 57150, 51136, 28536, 18838.4],
    "DMC_Rate_Pct"     : [2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 1.0],
    "DMC_Rs"           : [16220, 26700, 26742, 21512, 22860, 25568, 14268, 4709.6],
}
df = pd.DataFrame(portfolio_data)

# Step 3 & 4 aggregates
total_scanning_risk = df["Scanning_Risk_Rs"].sum()
total_ic_credit     = 10714.05          # Guar Seed / Guar Gum spread credit
adj_scanning_risk   = total_scanning_risk - total_ic_credit
total_dmc           = df["DMC_Rs"].sum()
total_csc           = 0.0               # NIL — no cross-month positions
step4_margin        = adj_scanning_risk + total_dmc + total_csc

# ============================================================
# 5A. SHORT OPTION MINIMUM (SOM)
# ============================================================
# SOM is the FLOOR margin for portfolios with SHORT OPTIONS.
# It ensures that even if scanning risk is low (due to credits),
# a minimum margin is held for each short option lot.
#
# SOM = SOM_Rate (₹/lot) × Number of short option lots
#
# ── Portfolio Check ──────────────────────────────────────────
# This portfolio contains ONLY FUTURES — no options at all.
# ∴ Short Option Minimum = ₹ 0  (not applicable)
#
# If you add short options later, populate SOM_RATES below:
# SOM_RATES = { "SYMBOL" : som_rate_per_lot }
# ============================================================

SOM_RATES       = {}     # e.g. {"TMCFGRNZM_OPT": 5000}
short_opt_lots  = 0
total_som       = 0.0

# ============================================================
# 5B. NET OPTION VALUE (NOV)
# ============================================================
# NOV = market value of all option positions in the portfolio
#   Long  options → positive NOV  (asset — reduces margin)
#   Short options → negative NOV  (liability — adds to margin)
#
# Final SPAN Margin = max(Step4_Margin, SOM) − NOV  (floored at 0)
#
# ── Portfolio Check ──────────────────────────────────────────
# No options in portfolio → NOV = ₹ 0
# ============================================================

total_nov = 0.0

# ============================================================
# 5C. SPAN MARGIN (per position)
# ============================================================
# Apportion SPAN margin to each position proportionally by
# scanning risk contribution (standard industry approach)
# ============================================================

# --- Proportional SPAN margin per position ---
# Scanning risk share
df["SR_Share_Pct"]     = df["Scanning_Risk_Rs"] / total_scanning_risk

# Each position's share of IC credit
df["IC_Credit_Share"]  = df["SR_Share_Pct"] * total_ic_credit

# Position-level adjusted scanning risk
df["Adj_SR_Rs"]        = df["Scanning_Risk_Rs"] - df["IC_Credit_Share"]

# SPAN margin per position = Adj SR + DMC + CSC share (CSC=0 here)
df["SPAN_Margin_Rs"]   = df["Adj_SR_Rs"] + df["DMC_Rs"]

# ============================================================
# 5D. EXPOSURE MARGIN
# ============================================================
# MCX levies an Exposure Margin on top of SPAN margin.
# It acts as a second buffer against extreme/gap risk.
#
# MCX Standard Exposure Margin = 1% of Total Contract Value
# (Some commodities carry higher rates — check MCX circulars)
# ============================================================

EXPOSURE_MARGIN_PCT = 1.0    # % of total position value

df["Exposure_Margin_Rs"] = (EXPOSURE_MARGIN_PCT / 100) * df["Total_Exposure"]
total_exposure_margin    = df["Exposure_Margin_Rs"].sum()

# ============================================================
# 5E. TOTAL MARGIN REQUIRED PER POSITION
# ============================================================

df["Total_Margin_Rs"] = df["SPAN_Margin_Rs"] + df["Exposure_Margin_Rs"]

# ============================================================
# 5F. FINAL SPAN MARGIN — PORTFOLIO LEVEL
# ============================================================

raw_span        = step4_margin                         # Before SOM / NOV
span_vs_som     = max(raw_span, total_som)             # Floor at SOM
final_span      = max(span_vs_som - total_nov, 0.0)   # Subtract NOV, floor at 0
final_total     = final_span + total_exposure_margin   # + Exposure Margin

total_invested  = df["Total_Exposure"].sum()           # Actual capital deployed

# Margin utilization
margin_util_pct = (final_total / total_invested) * 100

# ============================================================
# 6. DISPLAY — FULL SUMMARY
# ============================================================

SEP = "=" * 72

print(SEP)
print("  SPAN — STEP 5 : FINAL MARGIN ASSEMBLY")
print(SEP)

# ── Per-position table ──────────────────────────────────────
print("\n📋 Per-Position Margin Breakdown:")
display_cols = [
    "Commodity", "Num_Lots", "Total_Exposure",
    "Scanning_Risk_Rs", "IC_Credit_Share", "Adj_SR_Rs",
    "DMC_Rs", "SPAN_Margin_Rs", "Exposure_Margin_Rs", "Total_Margin_Rs"
]
print(df[display_cols].to_string(index=False))

# ── Totals row ─────────────────────────────────────────────
print(f"\n{'─'*72}")
print(f"  {'TOTAL':40s}  ₹{df['Total_Margin_Rs'].sum():>14,.2f}")
print(f"{'─'*72}")

# ── SPAN Build-up ──────────────────────────────────────────
print(f"""
{'─'*60}
  SPAN MARGIN BUILD-UP
{'─'*60}
  [1] Raw Scanning Risk (16 scenarios)     : ₹{total_scanning_risk:>12,.2f}
  [2] (−) Inter-Commodity Spread Credit    : ₹{total_ic_credit:>12,.2f}
      └─ Guar Seed / Guar Gum  @ 75%
  [3] (−) Adjusted Scanning Risk           : ₹{adj_scanning_risk:>12,.2f}
  [4] (+) Delivery Month Charge (DMC)      : ₹{total_dmc:>12,.2f}
  [5] (+) Calendar Spread Charge (CSC)     : ₹{total_csc:>12,.2f}  [NIL]
{'─'*60}
  [6] Step 4 Margin  (3+4+5)              : ₹{step4_margin:>12,.2f}
  [7] Short Option Minimum (SOM)           : ₹{total_som:>12,.2f}  [NIL]
  [8] SPAN Margin  max([6],[7])            : ₹{span_vs_som:>12,.2f}
  [9] (−) Net Option Value (NOV)           : ₹{total_nov:>12,.2f}  [NIL]
{'─'*60}
  [A] Final SPAN Margin                    : ₹{final_span:>12,.2f}
  [B] (+) Exposure Margin  @ {EXPOSURE_MARGIN_PCT:.1f}%            : ₹{total_exposure_margin:>12,.2f}
{'─'*60}
  [C] TOTAL MARGIN REQUIRED                : ₹{final_total:>12,.2f}
{'─'*60}
""")

# ── Portfolio Health Check ─────────────────────────────────
print("📊 Portfolio Health Check:")
print(f"{'─'*60}")
print(f"  Total Portfolio Exposure    : ₹{total_invested:>14,.2f}")
print(f"  Total Margin Required       : ₹{final_total:>14,.2f}")
print(f"  SPAN Margin Only            : ₹{final_span:>14,.2f}")
print(f"  Exposure Margin             : ₹{total_exposure_margin:>14,.2f}")
print(f"  Margin as % of Exposure     :  {margin_util_pct:>12.2f}%")
print(f"  Leverage Implied            :  {total_invested/final_total:>12.2f}x")
print(f"{'─'*60}")

# ── Sector-wise Margin ─────────────────────────────────────
print("\n📂 Sector-wise Margin Summary:")
sector_df = (
    df.groupby("Sector")
    .agg(
        Total_Exposure   = ("Total_Exposure",    "sum"),
        SPAN_Margin      = ("SPAN_Margin_Rs",    "sum"),
        Exposure_Margin  = ("Exposure_Margin_Rs","sum"),
        Total_Margin     = ("Total_Margin_Rs",   "sum"),
    )
    .reset_index()
    .assign(Margin_Pct = lambda x: (x["Total_Margin"] / x["Total_Exposure"] * 100).round(2))
    .sort_values("Total_Margin", ascending=False)
)
print(sector_df.to_string(index=False))

# ── SOM / NOV note ─────────────────────────────────────────
print(f"""
💡 Notes:
  • SOM  = NIL  → Portfolio is futures-only (no short options)
  • NOV  = NIL  → No option positions in portfolio
  • DMC  applied at 2% (contracts 6–15 days to expiry, Apr 20)
              and 1% (Kapas, 22 days to expiry, Apr 30)
  • Exposure Margin @ {EXPOSURE_MARGIN_PCT:.1f}% is charged by MCX in addition to SPAN
  • Margin requirement will INCREASE daily as expiry approaches
    (DMC tier will step up: 2% → 3% in last 5 days)
""")

# ============================================================
# 7. FINAL OUTPUT  (complete SPAN result)
# ============================================================

span_final = {
    "position_df"           : df,
    "total_scanning_risk"   : total_scanning_risk,
    "total_ic_credit"       : total_ic_credit,
    "adj_scanning_risk"     : adj_scanning_risk,
    "total_dmc"             : total_dmc,
    "total_csc"             : total_csc,
    "total_som"             : total_som,
    "total_nov"             : total_nov,
    "final_span_margin"     : final_span,
    "total_exposure_margin" : total_exposure_margin,
    "total_margin_required" : final_total,
    "total_exposure"        : total_invested,
    "margin_util_pct"       : margin_util_pct,
}

print("✅ SPAN Calculation Complete!")
print(f"   `span_final` dict contains all results.")
print(f"   Keys: {list(span_final.keys())}")

  SPAN — STEP 5 : FINAL MARGIN ASSEMBLY

📋 Per-Position Margin Breakdown:
           Commodity  Num_Lots  Total_Exposure  Scanning_Risk_Rs  IC_Credit_Share  Adj_SR_Rs    DMC_Rs  SPAN_Margin_Rs  Exposure_Margin_Rs  Total_Margin_Rs
            Turmeric         1          811000         40,550.00         1,132.59  39,417.41 16,220.00       55,637.41            8,110.00        63,747.41
           Coriander         2         1335000         66,750.00         1,864.37  64,885.63 26,700.00       91,585.63           13,350.00       104,935.63
               Jeera         2         1337100         66,855.00         1,867.30  64,987.70 26,742.00       91,729.70           13,371.00       105,100.70
            Guar Gum         2         1075600         53,780.00         1,502.11  52,277.89 21,512.00       73,789.89           10,756.00        84,545.89
           Guar Seed         4         1143000         57,150.00         1,596.23  55,553.77 22,860.00       78,413.77           11,430.00        